# Averaged Waveforms Pipeline

This notebook loads the `all_waveforms` dictionary produced by `main_pipeline.ipynb`, bins waveforms by:
- **Frequency** (Hz): `15–25`, `25–35`, `35–45`, `45+`
- **Speed**: `fast` (median spiking > 35 Hz) or `slow`
- **Signal type**: `Excitatory` or `Inhibitory`

Two frequency methods are supported:
| Method | Description |
|---|---|
| `annotation` | `freq = 1 / (next_annotation_time − current_annotation_time)` — uses raw annotation interval |
| `blackdot` | `freq = 1 / (next_BlackDotTime − current_BlackDotTime)` — burst-to-burst via black dots |
| `midpoint` | `freq = 1 / (next_Midpoint − current_Midpoint)` — interval between midpoints of consecutive black-dot pairs |

Set `FREQ_METHOD` in **Section 3** to choose.  
Output CSV is written to `data/processed/`.

## 1 · Import Required Libraries

In [1]:
import sys, os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ── make sure the scripts package is importable ──────────────────────────────
REPO_ROOT = Path(os.getcwd()).parents[1]   # .../murray-neuroscience-lab
sys.path.insert(0, str(REPO_ROOT / "analysis" / "scripts"))

from waveforms import bin_wave

## 2 · Load Waveforms Dictionary

In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
WAVEFORMS_PKL = PROCESSED_DIR / "all_waveforms.pkl"

with open(WAVEFORMS_PKL, "rb") as f:
    all_waveforms = pickle.load(f)

is_flat_dict = bool(all_waveforms) and isinstance(next(iter(all_waveforms.keys())), tuple)

if is_flat_dict:
    total_traces = "legacy-flat"
    total_waveforms = len(all_waveforms)
    print("Loaded legacy flat waveform dictionary.")
else:
    total_traces = sum(len(v) for v in all_waveforms.values())
    total_waveforms = sum(
        len(wv)
        for cell_data in all_waveforms.values()
        for wv in cell_data.values()
    )
    print("Loaded nested waveform dictionary.")

print(f"Cells loaded   : {len(all_waveforms) if not is_flat_dict else 'n/a'}")
print(f"Traces loaded  : {total_traces}")
print(f"Waveforms total: {total_waveforms}")

sample_key = next(iter(all_waveforms)) if is_flat_dict else next(iter(all_waveforms[next(iter(all_waveforms))][next(iter(all_waveforms[next(iter(all_waveforms))]))]))
print(f"\nSample key (freq, signal_type, median, mean):\n  {sample_key}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/Haley/Desktop/murray-neuroscience-lab/data/processed/all_waveforms.pkl'

## 3 · Configuration

Set `FREQ_METHOD` to one of:
- `"annotation"` — uses the raw interval between consecutive annotation times (already stored in the waveform key)
- `"blackdot"` — recomputes frequency from `BlackDotTime` column (`1 / blackdot_interval`)
- `"midpoint"` — recomputes frequency from `Midpoint` column (`1 / midpoint_interval`) — **matches** `averaged_waveforms_by_freq_speed_signaltype_median.csv`

In [ ]:
# ── user-tunable options ──────────────────────────────────────────────────────

# Frequency method: "annotation" | "blackdot" | "midpoint"
FREQ_METHOD = "midpoint"

# Phase bins per waveform
NUM_BINS = 100

# Speed cutoff (Hz): cells with median spiking > cutoff → "fast", else "slow"
SPEED_CUTOFF = 35.0

# Frequency bin edges and labels
FREQ_BINS   = [15, 25, 35, 45, np.inf]
FREQ_LABELS = ["15-25", "25-35", "35-45", "45+"]

# Signal types to keep (cell-attached excluded)
KEEP_SIGNAL_TYPES = {"Excitatory", "Inhibitory"}

print(f"Frequency method : {FREQ_METHOD}")
print(f"Phase bins       : {NUM_BINS}")
print(f"Speed cutoff     : {SPEED_CUTOFF} Hz")

## 4 · Aggregation Helper Functions

In [ ]:
def get_freq_bin(freq, bins=FREQ_BINS, labels=FREQ_LABELS):
    """Map a scalar frequency (Hz) to a bin label, or None if out of range."""
    for lo, hi, label in zip(bins[:-1], bins[1:], labels):
        if lo <= freq < hi:
            return label
    return None


def get_speed_bin(median_spiking, cutoff=SPEED_CUTOFF):
    """Return 'fast' or 'slow' based on median spiking rate."""
    if pd.isna(median_spiking):
        return None
    return "fast" if float(median_spiking) > cutoff else "slow"


def normalize_signal_type(s, keep=KEEP_SIGNAL_TYPES):
    """Return the signal type string if it's one we keep, else None."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    return s if s in keep else None


def compute_midpoints_and_blackdot_freq(annotations_df):
    """
    Given an annotations DataFrame that already has a BlackDotTime column,
    return a copy with Midpoint and Freq_Blackdots columns added.
    Mirrors calculate_midpoints_and_frequencies() from utils.py.
    """
    df = annotations_df.copy().reset_index(drop=True)
    df["Midpoint"] = np.nan
    df["Freq_Blackdots"] = np.nan
    df["Freq_Midpoints"] = np.nan

    for i in range(len(df)):
        if i + 1 == len(df):
            # last row: mirror the half-interval
            if i > 0 and not pd.isna(df.loc[i - 1, "Midpoint"]):
                half = df.loc[i, "BlackDotTime"] - df.loc[i - 1, "Midpoint"]
                df.loc[i, "Midpoint"] = df.loc[i, "BlackDotTime"] + half
        else:
            interval_bd = df.loc[i + 1, "BlackDotTime"] - df.loc[i, "BlackDotTime"]
            if interval_bd > 0:
                df.loc[i + 1, "Freq_Blackdots"] = 1.0 / interval_bd
            midpoint = df.loc[i, "BlackDotTime"] + interval_bd / 2.0
            df.loc[i, "Midpoint"] = midpoint

    # Freq_Midpoints: 1 / (midpoint[i+1] - midpoint[i])
    for i in range(len(df) - 1):
        mp_curr = df.loc[i, "Midpoint"]
        mp_next = df.loc[i + 1, "Midpoint"]
        if not pd.isna(mp_curr) and not pd.isna(mp_next):
            interval_mp = mp_next - mp_curr
            if interval_mp > 0:
                df.loc[i, "Freq_Midpoints"] = 1.0 / interval_mp

    return df


# ── map FREQ_METHOD → which frequency column / key index to use ───────────────
FREQ_COLUMN_MAP = {
    "annotation": None,          # already in waveform key[0]
    "blackdot":   "Freq_Blackdots",
    "midpoint":   "Freq_Midpoints",
}

print("Helper functions defined.")

## 5 · Build Averaged Waveforms DataFrame

For each waveform in the dict:
1. Resolve the frequency using the chosen method
2. Assign freq bin, speed bin, and signal type
3. Bin the waveform into `NUM_BINS` phase bins
4. Collect all rows → group by `(freq_bin, speed, signal_type, phase_bin)` → mean

In [ ]:
def build_averaged_waveforms(all_waveforms, freq_method=FREQ_METHOD,
                              num_bins=NUM_BINS, sheets=None):
    """
    Flatten all_waveforms into a long-form DataFrame, bin each waveform,
    then average within (freq_bin, speed, signal_type, Phase) groups.
    Supports both the legacy flat dict and the nested refactor dict.
    """
    freq_col = FREQ_COLUMN_MAP.get(freq_method)
    is_flat_dict = bool(all_waveforms) and isinstance(next(iter(all_waveforms.keys())), tuple)

    if is_flat_dict and freq_method != "midpoint":
        print(f"⚠️  Loaded legacy flat dict: key frequencies already reflect the midpoint-based legacy pipeline.")
        print(f"   Requested method='{freq_method}' cannot be recomputed from flat dict; using stored key frequency instead.")
        freq_col = None

    need_annotation_lookup = (not is_flat_dict) and freq_col is not None and sheets is not None
    all_rows = []

    if is_flat_dict:
        waveform_items = all_waveforms.items()
        for key, wave_df in waveform_items:
            raw_freq, signal_type, median_spike, mean_spike = key
            freq = raw_freq
            freq_bin = get_freq_bin(freq)
            speed_bin = get_speed_bin(median_spike)
            sig_type = normalize_signal_type(signal_type)
            if freq_bin is None or speed_bin is None or sig_type is None:
                continue
            if wave_df.empty or "Phase" not in wave_df.columns:
                continue
            binned = bin_wave(wave_df.copy(), num_bins=num_bins)
            binned["freq bin"] = freq_bin
            binned["signal type"] = sig_type
            binned["speed population"] = speed_bin
            all_rows.append(binned)
    else:
        for cell_name, traces in all_waveforms.items():
            enriched_freq_lookup = {}
            if need_annotation_lookup and cell_name in sheets:
                cell_sheet = sheets[cell_name]
                annotations = (cell_sheet["annotations"] if isinstance(cell_sheet, dict) else cell_sheet)
                if "BlackDotTime" in annotations.columns:
                    enriched = compute_midpoints_and_blackdot_freq(annotations)
                    for trace_name in traces:
                        trace_rows = enriched[enriched["Trace name"] == trace_name].reset_index(drop=True)
                        for idx in range(len(trace_rows)):
                            enriched_freq_lookup[(trace_name, idx)] = trace_rows.loc[idx, freq_col]

            for trace_name, waveforms in traces.items():
                for row_idx, (key, wave_df) in enumerate(waveforms.items()):
                    raw_freq, signal_type, median_spike, mean_spike = key
                    if freq_col is None or not need_annotation_lookup:
                        freq = raw_freq
                    else:
                        freq = enriched_freq_lookup.get((trace_name, row_idx), np.nan)
                        if pd.isna(freq):
                            freq = raw_freq

                    freq_bin = get_freq_bin(freq)
                    speed_bin = get_speed_bin(median_spike)
                    sig_type = normalize_signal_type(signal_type)
                    if freq_bin is None or speed_bin is None or sig_type is None:
                        continue
                    if wave_df.empty or "Phase" not in wave_df.columns:
                        continue
                    binned = bin_wave(wave_df.copy(), num_bins=num_bins)
                    binned["freq bin"] = freq_bin
                    binned["signal type"] = sig_type
                    binned["speed population"] = speed_bin
                    all_rows.append(binned)

    if not all_rows:
        print("⚠️  No rows collected — check your data or column names.")
        return pd.DataFrame(), pd.DataFrame()

    rows_df = pd.concat(all_rows, ignore_index=True)
    averaged_df = (
        rows_df
        .groupby(["freq bin", "signal type", "speed population", "Phase"], observed=True)["Normalized Current"]
        .mean()
        .reset_index()
    )

    print(f"Total binned rows  : {len(rows_df):,}")
    print(f"Averaged rows      : {len(averaged_df):,}")
    print(f"Groups             : {averaged_df[['freq bin','signal type','speed population']].drop_duplicates().shape[0]}")
    return averaged_df, rows_df


try:
    with open(PROCESSED_DIR / "sheets.pkl", "rb") as f:
        sheets = pickle.load(f)
    print("Loaded sheets.pkl")
except FileNotFoundError:
    sheets = None

averaged_df, rows_df = build_averaged_waveforms(all_waveforms, freq_method=FREQ_METHOD, sheets=sheets)
averaged_df.head(10)

## 6 · Export to CSV

In [ ]:
if not averaged_df.empty:
    out_fname = f"averaged_waveforms_{FREQ_METHOD}_freq.csv"
    out_path  = PROCESSED_DIR / out_fname
    averaged_df.to_csv(out_path, index=False)
    print(f"Saved → {out_path}")
else:
    print("Nothing to save.")

## 7 · Visualize Averaged Waveforms

Grid of subplots: rows = frequency bins, columns = signal types.  
Fast and slow speed populations are overlaid in different colors.

In [ ]:
def plot_averaged_waveforms(averaged_df,
                            freq_labels=FREQ_LABELS,
                            signal_colors=None,
                            fig_title=None):
    """
    Plot averaged waveforms in a freq_bin × speed grid (4 × 2 = 8 subplots).
    Excitatory and Inhibitory signal types are overlaid in different colors.
    """
    if averaged_df.empty:
        print("No data to plot.")
        return

    if signal_colors is None:
        signal_colors = {"Excitatory": "#d62728", "Inhibitory": "#1f77b4"}

    speeds = ["fast", "slow"]
    freq_bins_present = [fb for fb in freq_labels
                         if fb in averaged_df["freq bin"].unique()]

    n_rows = len(freq_bins_present)   # up to 4
    n_cols = len(speeds)              # 2

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 3 * n_rows),
        sharey=False, sharex=True,
        squeeze=False
    )

    for r, fb in enumerate(freq_bins_present):
        for c, speed in enumerate(speeds):
            ax = axes[r][c]
            subset = averaged_df[
                (averaged_df["freq bin"]         == fb) &
                (averaged_df["speed population"] == speed)
            ]

            if subset.empty:
                ax.text(0.5, 0.5, "no data", ha="center", va="center",
                        transform=ax.transAxes, fontsize=9, color="gray")
            else:
                for sig_type, color in signal_colors.items():
                    sig_data = subset[subset["signal type"] == sig_type]
                    if not sig_data.empty:
                        sig_sorted = sig_data.sort_values("Phase")
                        ax.plot(sig_sorted["Phase"],
                                sig_sorted["Normalized Current"],
                                color=color, linewidth=1.8,
                                label=sig_type)

            ax.set_xlim(0, 1)
            ax.set_ylim(-0.05, 1.05)
            ax.set_xlabel("Phase" if r == n_rows - 1 else "")

            # Column header (speed)
            if r == 0:
                ax.set_title(speed.capitalize(), fontsize=11, fontweight="bold")

            # Row label (freq bin)
            if c == 0:
                ax.set_ylabel(f"{fb} Hz\nNorm. Current", fontsize=9)

            # Legend on top-right subplot only
            if r == 0 and c == n_cols - 1:
                ax.legend(title="Signal type", fontsize=8, loc="upper right")

    suptitle = fig_title or f"Averaged Waveforms  |  freq method: {FREQ_METHOD}"
    fig.suptitle(suptitle, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
    return fig


fig = plot_averaged_waveforms(averaged_df)


### 7b · Compare Methods Side-by-Side (Optional)

Run this cell to generate averaged waveforms for **all three** frequency methods and compare them visually. This takes a bit longer since it re-runs the aggregation three times.

In [ ]:
# Pick a single (freq_bin, signal_type, speed) combination to compare side-by-side
COMPARE_FREQ_BIN   = "25-35"
COMPARE_SIGNAL     = "Excitatory"
COMPARE_SPEED      = "fast"

method_colors = {
    "annotation": "#2ca02c",
    "blackdot":   "#ff7f0e",
    "midpoint":   "#9467bd",
}

fig, ax = plt.subplots(figsize=(7, 4))

for method, color in method_colors.items():
    df_m, _ = build_averaged_waveforms(all_waveforms, freq_method=method, sheets=sheets)
    subset = df_m[
        (df_m["freq bin"]         == COMPARE_FREQ_BIN) &
        (df_m["signal type"]      == COMPARE_SIGNAL)   &
        (df_m["speed population"] == COMPARE_SPEED)
    ].sort_values("Phase")
    if not subset.empty:
        ax.plot(subset["Phase"], subset["Normalized Current"],
                color=color, linewidth=2, label=method)

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Phase")
ax.set_ylabel("Normalized Current")
ax.set_title(f"Freq-method comparison — {COMPARE_FREQ_BIN} Hz | {COMPARE_SIGNAL} | {COMPARE_SPEED}")
ax.legend(title="Freq method")
plt.tight_layout()
plt.show()